In [1]:
# Step 7: Setup
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI

# Load the API key from the .env file
load_dotenv()
print("API key loaded:", os.getenv("OPENAI_API_KEY") is not None)

# Create the OpenAI client and choose the model
client = OpenAI()
MODEL = "gpt-4o-mini"

# Quick test call
test = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in one word."}],
)
print(test.choices[0].message.content)

API key loaded: True
Hello!


In [6]:
# Step 8: Implement the LLM-as-judge (updated: also receives the expected answer)

JUDGE_PROMPT = """You are an expert quality reviewer for NordHome's customer service team.
Your job is to grade a chatbot's reply to a customer. Be strict and fair.

### TASK DESCRIPTION
The chatbot received a message from a customer and had to reply using ONLY the NordHome policy below.
A good reply solves the customer's problem within the policy, in a tone that fits the situation.

NordHome policy:
{policy}

Customer message:
{customer_message}

Expected correct answer (written by the evaluation team):
{ground_truth}

Chatbot reply:
{chatbot_response}

### EVALUATION CRITERIA
1. empathy_tone: The reply acknowledges the customer's specific situation (e.g. the damage, the waiting,
   the worry), apologises where something went wrong, and stays calm and polite. It is not robotic,
   defensive, or blaming the customer.
2. policy_accuracy: The reply gives the correct answer described in the expected correct answer,
   including all key facts it lists. Any policy fact it mentions must be correct.
3. no_invented_promises: The reply does NOT offer or state anything outside the policy and catalogue
   (no vouchers, discounts, compensation, exceptions, or made-up product facts).
   If the customer asks for something the policy does not allow, the reply politely says no.

### REASONING STEPS
Step 1: Read the policy and the expected correct answer. List the key facts the reply must contain.
Step 2: Check policy_accuracy: does the reply give the expected answer, and is every policy fact correct?
Step 3: Check no_invented_promises: does the reply promise or state anything not in the policy? Quote it if so.
Step 4: Check empathy_tone: does the reply acknowledge the specific situation and stay polite without being defensive?
        Do NOT reward length: a short reply can score high if it covers everything.
Step 5: Decide each criterion true/false, then give the overall score using the scale below.

### SCORE SCALE
5 = All 3 criteria met; reply is clear and ready to send as-is.
4 = All 3 criteria met, but with a small weakness (slightly generic apology, unclear next step).
3 = One criterion not met, but nothing harmful (e.g. forgets one key fact such as the return fee).
2 = Two criteria not met, OR the tone is clearly defensive/robotic.
1 = Invents a promise outside the policy, OR states wrong policy facts, OR is rude.
An invented promise always means a score of 1, even if the rest is excellent.

### OUTPUT FORMAT
Return ONLY valid JSON in this exact format:
{
  "score": <integer 1-5>,
  "reasoning": "<2-4 sentences explaining the score, quoting the reply where useful>",
  "criteria_met": {
    "empathy_tone": true/false,
    "policy_accuracy": true/false,
    "no_invented_promises": true/false
  }
}"""


def judge_response(customer_message, ground_truth, chatbot_response):
    # Fill the placeholders in the judge prompt
    prompt = JUDGE_PROMPT.replace("{policy}", POLICY)
    prompt = prompt.replace("{customer_message}", customer_message)
    prompt = prompt.replace("{ground_truth}", ground_truth)
    prompt = prompt.replace("{chatbot_response}", chatbot_response)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"},
    )

    # Turn the JSON text into a Python dictionary
    try:
        result = json.loads(response.choices[0].message.content)
    except json.JSONDecodeError:
        result = {"score": None, "reasoning": "Judge did not return valid JSON", "criteria_met": {}}

    # Keep the token count for the cost calculation later
    result["input_tokens"] = response.usage.prompt_tokens
    result["output_tokens"] = response.usage.completion_tokens
    return result


# Test with one simple example: a reply that invents a voucher (should score 1)
test_message = "My Fjord sofa arrived with a tear. I want a refund and compensation for my lost day off!"
test_truth = "Apologise, offer free replacement or full refund with free return shipping, politely say no extra compensation."
test_reply = "So sorry about that! We'll send you a full refund and a €50 voucher for your trouble."

test_result = judge_response(test_message, test_truth, test_reply)
print(json.dumps(test_result, indent=2))

{
  "score": 1,
  "reasoning": "The reply offers a \u20ac50 voucher for the customer's trouble, which is an invented promise not allowed by the policy. The correct response should have offered a full refund or a free replacement without any additional compensation. This significant deviation from the policy results in a score of 1.",
  "criteria_met": {
    "empathy_tone": true,
    "policy_accuracy": false,
    "no_invented_promises": false
  },
  "input_tokens": 951,
  "output_tokens": 105
}


In [7]:
# Step 9: Create test dataset

test_cases = [
    {
        "id": 1,
        "title": "Standard return within the window",
        "prompt": "Hi, my Oslo Oak Desk was delivered 12 days ago. I haven't opened the box. How do I return it and when do I get my money back?",
        "ground_truth": "Return is allowed (within 30 days, unused). €6.95 return shipping is deducted from the refund. Refund goes to the original payment method within 5-7 business days after the desk reaches the warehouse.",
        "expected_criteria": ["confirms return is allowed", "mentions €6.95 fee", "mentions 5-7 business days"],
    },
    {
        "id": 2,
        "title": "Used mattress return",
        "prompt": "I bought a mattress three weeks ago and slept on it, but it's too firm for me. I want to return it. You have a 30-day return policy, right?",
        "ground_truth": "Return is NOT allowed: mattresses cannot be returned once used, even within 30 days. Say no clearly and politely. No exceptions, partial refunds or exchanges.",
        "expected_criteria": ["refuses the return", "explains used mattresses can't be returned", "no invented exception"],
    },
    {
        "id": 3,
        "title": "Lamp wattage + warranty",
        "prompt": "Can I put a 60 W bulb in the Linnea Table Lamp? And how long is the warranty on it?",
        "ground_truth": "No, the maximum is 40 W. The warranty is not in the catalogue: say this information is not available and offer to check with the team. Do not state any warranty period.",
        "expected_criteria": ["says max 40 W", "does not invent a warranty", "offers to check"],
    },
    {
        "id": 4,
        "title": "Angry complaint about damaged sofa",
        "prompt": "THIS IS RIDICULOUS. I waited 3 weeks for my Fjord sofa and it arrived with a huge tear in the fabric. I took a day off work for this delivery. I want a full refund AND compensation for my lost day, or I'm leaving a 1-star review everywhere.",
        "ground_truth": "Apologise sincerely and acknowledge the frustration. Offer a free replacement or a full refund, with free return shipping. Politely say no extra compensation can be offered.",
        "expected_criteria": ["sincere apology", "offers replacement or refund + free shipping", "no compensation promised"],
    },
    {
        "id": 5,
        "title": "Safety incident with legal threat",
        "prompt": "The Linnea lamp started sparking last night and burned my son's hand when he touched it. I'm talking to my lawyer. What's wrong with this lamp?",
        "ground_truth": "Show concern, tell the customer to stop using the lamp, do not troubleshoot or guess the cause, do not admit or deny fault, and say exactly: 'I am transferring you to a human specialist.'",
        "expected_criteria": ["tells customer to stop using it", "no troubleshooting", "exact escalation phrase"],
    },
]


def generate_response(customer_message):
    # The chatbot: gpt-4o-mini with the NordHome policy as system prompt
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": POLICY},
            {"role": "user", "content": customer_message},
        ],
        temperature=0,
    )
    return response.choices[0].message.content


# Generate a chatbot reply for each test case
for case in test_cases:
    case["model_response"] = generate_response(case["prompt"])
    print(f"--- Case {case['id']}: {case['title']} ---")
    print(case["model_response"])
    print()

--- Case 1: Standard return within the window ---
You can return your Oslo Oak Desk since it is unused and in its original packaging. To initiate the return, please contact our customer service to arrange the return shipping. The return shipping cost is €6.95, which will be deducted from your refund.

Once the item reaches our warehouse, your refund will be processed to the original payment method within 5-7 business days.

--- Case 2: Used mattress return ---
Yes, we have a 30-day return policy, but mattresses cannot be returned once used. Unfortunately, you won't be able to return the mattress. If you have any other questions or need assistance, feel free to ask!

--- Case 3: Lamp wattage + warranty ---
The Linnea Table Lamp has a maximum wattage of 40 W, so you cannot put a 60 W bulb in it. I don't have information on the warranty for the lamp. Would you like me to check with the team?

--- Case 4: Angry complaint about damaged sofa ---
I am sorry to hear about the issue with your F

In [8]:
# Step 10: Run evaluation & collect metrics

# gpt-4o-mini prices in USD per 1 million tokens
PRICE_INPUT = 0.15
PRICE_OUTPUT = 0.60

results = []
total_start = time.time()

for case in test_cases:
    start = time.time()
    judgement = judge_response(case["prompt"], case["ground_truth"], case["model_response"])
    seconds = round(time.time() - start, 2)

    cost = (judgement["input_tokens"] * PRICE_INPUT + judgement["output_tokens"] * PRICE_OUTPUT) / 1_000_000

    results.append({
        "id": case["id"],
        "title": case["title"],
        "prompt": case["prompt"],
        "ground_truth": case["ground_truth"],
        "expected_criteria": case["expected_criteria"],
        "model_response": case["model_response"],
        "score": judgement["score"],
        "reasoning": judgement["reasoning"],
        "criteria_met": judgement["criteria_met"],
        "seconds": seconds,
        "input_tokens": judgement["input_tokens"],
        "output_tokens": judgement["output_tokens"],
        "cost_usd": round(cost, 6),
    })
    print(f"Case {case['id']} done: score {judgement['score']} ({seconds}s)")

total_seconds = round(time.time() - total_start, 2)

# Aggregate statistics
scores = [r["score"] for r in results if r["score"] is not None]
summary = {
    "model": MODEL,
    "number_of_cases": len(results),
    "average_score": round(sum(scores) / len(scores), 2),
    "min_score": min(scores),
    "max_score": max(scores),
    "total_seconds": total_seconds,
    "total_input_tokens": sum(r["input_tokens"] for r in results),
    "total_output_tokens": sum(r["output_tokens"] for r in results),
    "total_cost_usd": round(sum(r["cost_usd"] for r in results), 6),
}

print()
print(json.dumps(summary, indent=2))

# Save everything to a JSON file
with open("evaluation_results.json", "w") as f:
    json.dump({"summary": summary, "results": results}, f, indent=2, ensure_ascii=False)

print("Saved to evaluation_results.json")

Case 1 done: score 5 (2.36s)
Case 2 done: score 5 (1.83s)
Case 3 done: score 5 (2.43s)
Case 4 done: score 5 (1.75s)
Case 5 done: score 1 (2.11s)

{
  "model": "gpt-4o-mini",
  "number_of_cases": 5,
  "average_score": 4.2,
  "min_score": 1,
  "max_score": 5,
  "total_seconds": 10.47,
  "total_input_tokens": 5052,
  "total_output_tokens": 589,
  "total_cost_usd": 0.001112
}
Saved to evaluation_results.json


In [11]:
# Step 11: Analyze & summarize results

print("=" * 60)
print("DETAILED RESULTS")
print("=" * 60)

for r in results:
    met = sum(1 for value in r["criteria_met"].values() if value)
    total = len(r["criteria_met"])
    print(f"\nCase {r['id']}: {r['title']}")
    print(f"  Score: {r['score']}/5 | Criteria met: {met}/{total} | Time: {r['seconds']}s | Cost: ${r['cost_usd']}")
    for name, value in r["criteria_met"].items():
        print(f"    - {name}: {'yes' if value else 'NO'}")
    print(f"  Judge reasoning: {r['reasoning'][:300]}")

print("\n" + "=" * 60)
print("SCORE DISTRIBUTION")
print("=" * 60)
for s in [5, 4, 3, 2, 1]:
    count = scores.count(s)
    print(f"  Score {s}: {'#' * count} ({count})")

print("\n" + "=" * 60)
print("CRITERIA PERFORMANCE (how many cases met each criterion)")
print("=" * 60)
for name in ["empathy_tone", "policy_accuracy", "no_invented_promises"]:
    count = sum(1 for r in results if r["criteria_met"].get(name))
    print(f"  {name}: {count}/{len(results)}")

print("\n" + "=" * 60)
print("TIME & COST")
print("=" * 60)
print(f"  Total time: {summary['total_seconds']}s (avg {round(summary['total_seconds'] / len(results), 2)}s per case)")
print(f"  Total tokens: {summary['total_input_tokens']} input + {summary['total_output_tokens']} output")
print(f"  Total judge cost: ${summary['total_cost_usd']}")

DETAILED RESULTS

Case 1: Standard return within the window
  Score: 5/5 | Criteria met: 3/3 | Time: 2.36s | Cost: $0.000236
    - empathy_tone: yes
    - policy_accuracy: yes
    - no_invented_promises: yes
  Judge reasoning: The reply accurately addresses the customer's situation by confirming that the Oslo Oak Desk can be returned since it is unused and in original packaging. It correctly states the return shipping cost of €6.95, which will be deducted from the refund, and mentions that the refund will be processed wit

Case 2: Used mattress return
  Score: 5/5 | Criteria met: 3/3 | Time: 1.83s | Cost: $0.000208
    - empathy_tone: yes
    - policy_accuracy: yes
    - no_invented_promises: yes
  Judge reasoning: The reply accurately states that mattresses cannot be returned once used, which aligns with the policy. It maintains a polite tone and acknowledges the customer's situation without being defensive. There are no invented promises or incorrect policy facts present in the respo